In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.core.pylabtools import figsize
import plotly.subplots as sp
import plotly.graph_objects as go
import plotly.express as px


## Entropia Widmowa

In [4]:
def calculate_spectral_entropy(ampls, freqs, band=(1, 50)):

    mask = (freqs >= band[0]) & (freqs <= band[1])
    ampls_band = ampls[mask]

    psd = ampls_band ** 2

    psd_norm = psd / np.sum(psd)

    psd_norm = psd_norm[psd_norm > 0]

    entropy = -np.sum(psd_norm * np.log2(psd_norm))

    max_entropy = np.log2(len(psd_norm))
    normalized_entropy = entropy / max_entropy

    return normalized_entropy

def plot_fft(signal_df, channel, fs=200):
    data = signal_df[channel].values
    n = len(data)

    fft_values = np.fft.rfft(data)
    frequencies = np.fft.rfftfreq(n, d=1/fs)

    amplitude = np.abs(fft_values)

    return frequencies, amplitude

#### wyświetlenie entropii dla każdego pacjenta

In [11]:
mask_start = 1
mask_end = 30

for i in range(1, 20):
    if i < 10:
        i = "0" + str(i)
    signal_native = pd.read_csv(f"data/Filtered_Data/s{i}_ex05.csv")
    signal_non_native = pd.read_csv(f"data/Filtered_Data/s{i}_ex06.csv")
    signal_neutral = pd.read_csv(f"data/Filtered_Data/s{i}_ex07.csv")
    datasets = [signal_native, signal_non_native, signal_neutral]
    channels = ['P4', 'Cz', 'F8', 'T7']
    titles = ['Native', 'Non-Native', 'Neutral']

    entropy_results = {title: [] for title in titles}

    for col_idx, current_data in enumerate(datasets):
        for channel_name in channels:

            freqs, ampls = plot_fft(current_data, channel_name, fs=200)

            se = calculate_spectral_entropy(ampls, freqs, band=(mask_start, mask_end))
            entropy_results[titles[col_idx]].append(se)

    fig = go.Figure()

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    for idx, title in enumerate(titles):
        fig.add_trace(go.Bar(
            x=channels,
            y=entropy_results[title],
            name=title,
            marker_color=colors[idx],
            text=[f"{val:.3f}" for val in entropy_results[title]],
            textposition='auto'
        ))

    fig.update_layout(
        title_text=f"Porównanie Entropii Widmowej (Złożoności Sygnału) w paśmie {mask_start}-{mask_end} Hz",
        title_x=0.5,
        xaxis_title="Kanał EEG",
        yaxis_title="Znormalizowana Entropia Widmowa [0-1]",
        barmode='group', # Słupki obok siebie
        template="plotly_white",
        height=600,
        width=900,
        legend=dict(title="Eksperyment", x=1.02, y=1)
    )

    fig.show()

#### Wyświetlenie dla całej grupy

In [13]:
channels = ['P4', 'Cz', 'F8', 'T7']
titles = ['Native', 'Non-Native', 'Neutral']

all_results = []

print("Rozpoczynam analizę grupy...")

for i in range(1, 20):
    patient_id = f"s{i:02d}"

    try:
        # Wczytywanie danych
        signal_native = pd.read_csv(f"data/Filtered_Data/{patient_id}_ex05.csv")
        signal_non_native = pd.read_csv(f"data/Filtered_Data/{patient_id}_ex06.csv")
        signal_neutral = pd.read_csv(f"data/Filtered_Data/{patient_id}_ex07.csv")
        datasets = [signal_native, signal_non_native, signal_neutral]

        # Obliczenia dla konkretnego pacjenta
        for col_idx, current_data in enumerate(datasets):
            stan = titles[col_idx]
            for channel_name in channels:
                freqs, ampls = plot_fft(current_data, channel_name, fs=200)
                se = calculate_spectral_entropy(ampls, freqs, band=(mask_start, mask_end))

                # Zapisujemy pojedynczy wynik do listy
                all_results.append({
                    'Pacjent': patient_id,
                    'Kanał': channel_name,
                    'Stan': stan,
                    'Entropia': se
                })
    except FileNotFoundError:
        print(f"Brak pliku dla pacjenta {patient_id}. Pomijam.")

# Konwersja zebranych danych do wygodnej tabeli Pandas
df_results = pd.DataFrame(all_results)

# Tworzenie zbiorczego wykresu pudełkowego (Boxplot)
fig = px.box(
    df_results,
    x="Kanał",
    y="Entropia",
    color="Stan",
    title=f"Analiza Grupowa: Entropia Widmowa ({mask_start}-{mask_end} Hz) dla {len(df_results['Pacjent'].unique())} pacjentów",
    labels={"Entropia": "Znormalizowana Entropia Widmowa [0-1]"},
    template="plotly_white",
    points="all"
)

fig.update_layout(
    height=600,
    width=1000,
    legend=dict(title="Eksperyment"),
    boxmode='group'
)

fig.show()

Rozpoczynam analizę grupy...


## Entropia Przybliżona

#### wszyscy pacjenci

In [14]:

def calculate_approximate_entropy(U, m=2, r=None):
    if r is None:
        r = 0.2 * np.std(U)

    def _phi(m):
        N = len(U)
        x = np.array([U[i:i+m] for i in range(N - m + 1)])

        C = np.sum(np.max(np.abs(x[:, np.newaxis] - x[np.newaxis, :]), axis=2) <= r, axis=0) / (N - m + 1)
        return np.sum(np.log(C)) / (N - m + 1)

    return np.abs(_phi(m) - _phi(m + 1))


for i in range(1, 20):
    if i < 10:
        i = "0" + str(i)
    signal_native = pd.read_csv(f"data/Filtered_Data/s{i}_ex05.csv")
    signal_non_native = pd.read_csv(f"data/Filtered_Data/s{i}_ex06.csv")
    signal_neutral = pd.read_csv(f"data/Filtered_Data/s{i}_ex07.csv")
    datasets = [signal_native, signal_non_native, signal_neutral]
    channels = ['P4', 'Cz', 'F8', 'T7']
    titles = ['Native', 'Non-Native', 'Neutral']

    apen_results = {title: [] for title in titles}

    fs = 200
    window_samples = 2 * fs # 400 próbek


    for col_idx, current_data in enumerate(datasets):
        for channel_name in channels:

            signal_epoch = current_data[channel_name].values[1000 : 1000 + window_samples]

            apen_val = calculate_approximate_entropy(signal_epoch)
            apen_results[titles[col_idx]].append(apen_val)

    fig = go.Figure()

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

    for idx, title in enumerate(titles):
        fig.add_trace(go.Bar(
            x=channels,
            y=apen_results[title],
            name=title,
            marker_color=colors[idx],
            text=[f"{val:.3f}" for val in apen_results[title]],
            textposition='auto'
        ))

    fig.update_layout(
        title_text="Porównanie Entropii Przybliżonej (ApEn) na 2-sekundowych epokach EEG",
        title_x=0.5,
        xaxis_title="Kanał EEG",
        yaxis_title="Entropia Przybliżona (ApEn)",
        barmode='group',
        template="plotly_white",
        height=600,
        width=900,
        legend=dict(title="Eksperyment", x=1.02, y=1)
    )

    fig.show()

In [15]:

channels = ['P4', 'Cz', 'F8', 'T7']
titles = ['Native', 'Non-Native', 'Neutral']

fs = 200
window_samples = 2 * fs
start_sample = 1000

all_apen_results = []


for i in range(1, 20):
    patient_id = f"s{i:02d}"

    try:
        signal_native = pd.read_csv(f"data/Filtered_Data/{patient_id}_ex05.csv")
        signal_non_native = pd.read_csv(f"data/Filtered_Data/{patient_id}_ex06.csv")
        signal_neutral = pd.read_csv(f"data/Filtered_Data/{patient_id}_ex07.csv")

        datasets = [signal_native, signal_non_native, signal_neutral]

        for col_idx, current_data in enumerate(datasets):
            stan = titles[col_idx]

            for channel_name in channels:
                signal_epoch = current_data[channel_name].values[start_sample : start_sample + window_samples]

                apen_val = calculate_approximate_entropy(signal_epoch)

                all_apen_results.append({
                    'Pacjent': patient_id,
                    'Kanał': channel_name,
                    'Stan': stan,
                    'ApEn': apen_val
                })

        print(f"Przetworzono pacjenta {patient_id}")

    except FileNotFoundError:
        print(f"Brak pliku dla pacjenta {patient_id}. Pomijam.")

print("Obliczenia zakończone. Generuję wykres grupowy...")

df_apen_results = pd.DataFrame(all_apen_results)
num_patients = len(df_apen_results['Pacjent'].unique())

fig = px.box(
    df_apen_results,
    x="Kanał",
    y="ApEn",
    color="Stan",
    title=f"Analiza Grupowa: Entropia Przybliżona (ApEn) na 2-sekundowych epokach ({num_patients} pacjentów)",
    labels={"ApEn": "Entropia Przybliżona (ApEn)"},
    template="plotly_white",
    points="all" # Generuje kropki pokazujące pojedyncze wyniki
)

fig.update_layout(
    height=600,
    width=1000,
    legend=dict(title="Eksperyment"),
    boxmode='group'
)

fig.show()

Przetworzono pacjenta s01
Przetworzono pacjenta s02
Przetworzono pacjenta s03
Przetworzono pacjenta s04
Przetworzono pacjenta s05
Przetworzono pacjenta s06
Przetworzono pacjenta s07
Przetworzono pacjenta s08
Przetworzono pacjenta s09
Przetworzono pacjenta s10
Przetworzono pacjenta s11
Przetworzono pacjenta s12
Przetworzono pacjenta s13
Przetworzono pacjenta s14
Przetworzono pacjenta s15
Przetworzono pacjenta s16
Przetworzono pacjenta s17
Przetworzono pacjenta s18
Przetworzono pacjenta s19
Obliczenia zakończone. Generuję wykres grupowy...
